# Embeddings

We want to see whether the descriptions capture meaningful similarity between maintenance tasks. We compare `WODescription` alone with the same description plus system, subsystem, and component context.

The full embedding run has not been started. Set `RUN_EMBEDDINGS=True` when ready. The default is a local TF-IDF/SVD baseline; sentence-transformer embeddings are optional.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
helper_root = Path.cwd() if (Path.cwd() / 'pipeline_utils.py').exists() else Path.cwd() / 'fmucd_analysis'
if str(helper_root) not in sys.path:
    sys.path.insert(0, str(helper_root))
from pipeline_utils import (project_root, read_json, write_json, fingerprint,
                            assert_fingerprints, require_new_run, safe_run_name)
ROOT = project_root()
INPUT_DIR = ROOT / 'data/clustering'
source_manifest = read_json(INPUT_DIR / 'manifest.json')
SEED = 42
pd.set_option('display.max_colwidth', 100)

In [2]:
RUN_EMBEDDINGS = True
RUN_NAME = 'tfidf_review'
BACKEND = 'tfidf_svd'
OVERWRITE = True
FIT_TASK_LIMIT = 20_000
EMBEDDING_DIM = 128
MAX_VOCAB = 30_000
TRANSFORM_BATCH = 10_000
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
MODEL_REVISION = None
ALLOW_MODEL_DOWNLOAD = False
OUT_DIR = ROOT / 'data/embeddings' / safe_run_name(RUN_NAME)
VARIANTS = ['text_wo', 'text_context']

## 1. Prepare the text

We keep one text record per task and use training tasks to fit the text transformation. Later tasks are transformed without using their durations.

In [3]:
if RUN_EMBEDDINGS:
    import joblib
    import sklearn
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import Normalizer
    require_new_run(OUT_DIR, OVERWRITE)
    train_text = pd.read_parquet(INPUT_DIR / 'task_lookup_train.parquet', columns=['task_id'] + VARIANTS)
    later_text = pd.read_parquet(INPUT_DIR / 'work_order_features.parquet', columns=['task_id'] + VARIANTS)
    catalog = pd.concat([train_text, later_text], ignore_index=True).drop_duplicates('task_id', keep='first')
    catalog = catalog.sort_values('task_id').reset_index(drop=True)
    catalog['is_training_task'] = catalog['task_id'].isin(train_text['task_id'])
    assert catalog['task_id'].is_unique and train_text['task_id'].is_unique
    assert catalog[VARIANTS].notna().all().all()
    train_indices = np.flatnonzero(catalog['is_training_task'].to_numpy())
    rng = np.random.default_rng(SEED)
    fit_indices = (np.sort(rng.choice(train_indices, FIT_TASK_LIMIT, replace=False))
                   if FIT_TASK_LIMIT is not None and len(train_indices) > FIT_TASK_LIMIT else train_indices)

In [4]:
if RUN_EMBEDDINGS:
    catalog.to_parquet(OUT_DIR / 'catalog.parquet', index=False)
    pd.DataFrame({'task_id': catalog.iloc[fit_indices]['task_id']}).to_parquet(OUT_DIR / 'fit_task_ids.parquet', index=False)
    del later_text

## 2. Create the vectors

We fit each text variant separately and normalize its vectors. Batches keep memory use manageable. A semantic run needs `requirements-embeddings.txt` and a cached model or explicit download permission.

Before overwriting a run, close kernels that have its arrays memory-mapped, or choose a new `RUN_NAME`. The writer saves to a temporary file and closes it before replacing the output.

In [5]:
from pipeline_utils import write_vector_batches

In [6]:
if RUN_EMBEDDINGS:
    if BACKEND not in {'tfidf_svd', 'sentence_transformer'}:
        raise ValueError('Unknown embedding backend.')
    model = None
    if BACKEND == 'sentence_transformer':
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION,
            local_files_only=not ALLOW_MODEL_DOWNLOAD, trust_remote_code=False)
        model.save(str(OUT_DIR / 'sentence_transformer'))
    embedding_info = {}

In [7]:
if RUN_EMBEDDINGS:
    for variant in VARIANTS:
        transformer = None
        if BACKEND == 'tfidf_svd':
            vectorizer = TfidfVectorizer(max_features=MAX_VOCAB, min_df=1,
                ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
            sparse_fit = vectorizer.fit_transform(catalog.iloc[fit_indices][variant].tolist())
            dimensions = min(EMBEDDING_DIM, sparse_fit.shape[0]-1, sparse_fit.shape[1]-1)
            if dimensions < 2:
                raise ValueError('Not enough nonempty training text for SVD.')
            svd = TruncatedSVD(n_components=dimensions, random_state=SEED)
            svd.fit(sparse_fit)
            transformer = Pipeline([('tfidf', vectorizer), ('svd', svd), ('normalize', Normalizer())])
            joblib.dump(transformer, OUT_DIR / f'{variant}_transformer.joblib')
            del sparse_fit
        else:
            dimensions = model.get_sentence_embedding_dimension()
        if transformer is not None:
            encode = transformer.transform
        else:
            encode = lambda texts: model.encode(texts, batch_size=64,
                convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        embedding_info[variant] = write_vector_batches(
            OUT_DIR / f'{variant}.npy', catalog[variant], encode, dimensions, TRANSFORM_BATCH)

In [8]:
if RUN_EMBEDDINGS:
    source_files = ['manifest.json', 'task_lookup_train.parquet', 'work_order_features.parquet']
    embedding_manifest = {'stage': 'embeddings', 'run_name': RUN_NAME, 'backend': BACKEND,
        'seed': SEED, 'fit_task_count': len(fit_indices), 'fit_task_limit': FIT_TASK_LIMIT,
        'model_name': MODEL_NAME if model is not None else None, 'model_revision': MODEL_REVISION,
        'max_sequence_length': int(model.max_seq_length) if model is not None else None,
        'variants': embedding_info, 'source_fingerprints': fingerprint(INPUT_DIR / f for f in source_files),
        'versions': {'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__}}
    embedding_manifest['artifact_fingerprints'] = fingerprint(
        [OUT_DIR / 'catalog.parquet', OUT_DIR / 'fit_task_ids.parquet'] +
        [OUT_DIR / info['file'] for info in embedding_info.values()])
    write_json(OUT_DIR / 'manifest.json', embedding_manifest)
    display(pd.DataFrame(embedding_info).T)

,file,dimensions,rows,zero_vector_rows
text_wo,text_wo.npy,128,607391,299
text_context,text_context.npy,128,607391,0


## 3. Check similar tasks

We inspect nearest neighbors for both text inputs. Similar wording should reflect similar work scope; building names and generic phrases may create misleading matches.

In [9]:
if RUN_EMBEDDINGS:
    review_indices = np.sort(np.random.default_rng(SEED).choice(train_indices, min(2000, len(train_indices)), replace=False))
    for variant in VARIANTS:
        matrix = np.load(OUT_DIR / f'{variant}.npy', mmap_mode='r')
        sample = np.array(matrix[review_indices], copy=True)
        matrix._mmap.close()
        del matrix
        for query_position in range(min(2, len(sample))):
            similarities = sample @ sample[query_position]
            neighbors = np.argsort(-similarities)[:6]
            display(catalog.iloc[review_indices[neighbors]][['task_id', variant]].assign(
                cosine_similarity=similarities[neighbors]).reset_index(drop=True))

,task_id,text_wo,cosine_similarity
0,23134461607519426,CHECK ALL ROOF DRAINS AND EXTERIOR DRAINS M.L.S. - MEDPLEX ZONE,1.000000
1,5415853600209326328,PRE-DAYS OFF BUILDING CHECK HSRF MEDPLEX ZONE,0.758165
2,15226197451815329762,SULFUR ODOR C.R.F. 228 - MEDPLEX ZONE,0.676738
3,15910879606730098838,GIVEN BUIDLING ROUNDS 02/20-02/24/17 MEDPLEX ZONE,0.657097
4,9038572212043439093,FIX CARPET GIVEN C110 - MEDPLEX ZONE,0.640374
5,3877936508500941927,PRIME FLOOR DRAINS IN MECHANICAL RMS STAFFORD - MEDPLEX ZONE,0.637717


,task_id,text_wo,cosine_similarity
0,28500905587053303,TOO HOT IN BOOKSTORE,1.000000
1,4971247207997301832,*POB: 2.402 TOO HOT,0.964611
2,9083744703025204854,GAR: 2.206 TOO HOT,0.963995
3,12371905750482523212,"*SUT-3.114 / TOO HOT, T-STAT NOT RESPONDING",0.906491
4,3721859690027205494,"LAC: 3.412 / TOO HOT, T-STAT NOT RESPONDING.",0.905854
5,12985734582689567019,DENTAL - ROOM 5141 TOO HOT,0.902582


,task_id,text_context,cosine_similarity
0,23134461607519426,CHECK ALL ROOF DRAINS AND EXTERIOR DRAINS M.L.S. - MEDPLEX ZONE Plumbing Rain Water Drainage Roo...,1.000000
1,6373093307376197283,ZN1CARP - C BUILDING - ROOF TOP - NORTH - REPLACE DOWNSPOUT Plumbing Rain Water Drainage Roof Drain,0.832157
2,17569218221223342100,"HSRF, ROOM 006A, TROUBLESHOOT SUMP PUMP ISSUE-MEDPLEX ZONE Plumbing Rain Water Drainage Sump Pump",0.763203
3,15669060793218080794,UNB: NO POWER TO SUMP PUMP 007 Plumbing Rain Water Drainage Sump Pump,0.645899
4,4683667057467597124,219//STAIRWELL IN RAMP//NEED ELECTRICIAN TO VERIFY POWER TO CAM Plumbing Rain Water Drainage Pip...,0.583649
5,5898462176214843709,CHECK LOADING DOCK ENTRY - HSRF - MEDPLEX ZONE Roofing Roof Converings Concrete Roof,0.562139


,task_id,text_context,cosine_similarity
0,28500905587053303,TOO HOT IN BOOKSTORE HVAC Controls and Instrumentation HVAC Control Panel,1.000000
1,9083744703025204854,GAR: 2.206 TOO HOT HVAC Controls and Instrumentation HVAC Control Panel,0.985407
2,4971247207997301832,*POB: 2.402 TOO HOT HVAC Controls and Instrumentation HVAC Control Panel,0.984852
3,9746447288209239591,FC2: ALL THEIR OFFICES ARE TOO HOT NEED CORRECTED ASAP HVAC Controls and Instrumentation HVAC Co...,0.951074
4,15176100349350302196,BEG: WEST END - TOO HUMID / LOW AIR FLOW HVAC Controls and Instrumentation HVAC Control Panel,0.868817
5,5155415946113179128,CONTROL ISSUE HVAC Controls and Instrumentation HVAC Control Panel,0.851126


## What we will take forward

We will choose the text input after checking the neighbors. For the presentation, one description and a few useful neighbors are enough to explain the embedding step. The full matrices and fit-task IDs are saved for clustering.